# US Firm Characteristics: Risk Overlay Applicability

**Chapter 19 - Risk Management**

A risk overlay is a rule that closes a position on something the position does
while it is held: a stop-loss when it falls a set distance below entry, a
trailing stop when it falls that distance below its own high, a time exit after
a fixed number of bars. Every one of those rules asks what the price did
*between* the moment the position was opened and the moment it would otherwise
be closed.

This case study backtests on the vectorized forward-return path. That path
holds one weight vector per rebalance and multiplies it by the realized
forward return over the whole month; it never sees a price inside the month.
The information a stop needs is therefore not merely unused here, it is absent
from the data structure the backtest runs on. Simulating a stop on it would
mean inventing an intra-month path and reporting what the invention did.

So this notebook establishes a boundary rather than a result. It selects the
parent run the overlays would have been applied to, states which controls the
configuration declares, and registers none of them. The registry query in
section 3 is what confirms that: an empty result there is the outcome, not a
missing input.

**Learning Objectives:**
1. Select the parent run across the baseline and allocation stages
2. Decide whether a backtest path can represent a rule before configuring it
3. Separate a governance control from a validation variant that competes on Sharpe

**Book Reference:** Chapter 19, Sections 19.3-19.6, 19.8

**Prerequisites:** the Chapter 17 allocation sweep (`12_portfolio_management`),
whose runs are in `registry.db`.

In [ ]:
"""US Firm Characteristics: Risk: Engine-Level Risk Rules."""

import json
import time
import warnings

import polars as pl

warnings.filterwarnings("ignore")

from case_studies.utils.backtest_explorer import BacktestExplorer
from case_studies.utils.backtest_loaders import (
    VECTORIZED_CASE_STUDIES,
    get_backtest_config,
    load_backtest_prices_for,
)
from case_studies.utils.backtest_presets import (
    clone_backtest_spec,
    ensure_backtest_spec,
    strategy_view,
)
from case_studies.utils.backtest_runner import precompute_weights, run_backtest
from case_studies.utils.registry import read_predictions, resolve_best_backtest_runs
from case_studies.utils.sweep_config import (
    calibrate_trailing_stops,
    get_portfolio_risk_controls,
    get_position_risk_controls,
    get_top_n_predictions,
)
from utils.paths import get_case_study_dir

In [ ]:
CASE_STUDY_ID = "us_firm_characteristics"
LABEL = ""
# Reduces the price panel only. The vectorized path takes its universe and its P&L
# from the predictions frame and reads the panel for the rebalance calendar alone, so
# lowering this does not shrink a backtest here (agent-workspace #911). It is kept
# because the same parameter is what reduces the engine-path case studies.
MAX_SYMBOLS = 0
MAX_RISK_VARIANTS = 0  # 0 = all; >0 limits position + portfolio controls each
TOP_N_COMBOS = None

In [ ]:
CASE_DIR = get_case_study_dir(CASE_STUDY_ID)
bt_config = get_backtest_config(CASE_STUDY_ID)
if TOP_N_COMBOS is None:
    TOP_N_COMBOS = get_top_n_predictions(CASE_STUDY_ID, "risk_overlay")
if not LABEL:
    LABEL = bt_config.primary_label

IS_VECTORIZED = CASE_STUDY_ID in VECTORIZED_CASE_STUDIES
MODE_LABEL = "vectorized" if IS_VECTORIZED else "engine"
print(f"Case study: {CASE_STUDY_ID}, label: {LABEL}, mode: {MODE_LABEL}")

## 1. The Parent Run

An overlay is applied to something, so the first step is to say what. The
candidate is drawn from two stages at once: the equal-weight baselines from
`11_backtest` and the allocator variants from `12_portfolio_management`. Taking
the higher validation Sharpe of the two rather than always taking the allocator
keeps the funnel honest in the case where portfolio construction did not improve
on the equal-weight parent it was given.

The selection is on validation. The holdout is not read here and no overlay is
scored against it.

In [ ]:
def _resolve_pre_risk_runs(case_study: str, label: str, *, split: str, top_n: int) -> pl.DataFrame:
    candidates = [
        resolve_best_backtest_runs(
            case_study,
            label,
            split=split,
            stage=stage,
            top_n=top_n,
        )
        for stage in ("signal", "allocation")
    ]
    candidates = [frame for frame in candidates if not frame.is_empty()]
    if not candidates:
        return pl.DataFrame()
    return (
        pl.concat(candidates)
        .sort("sharpe", descending=True)
        .unique("backtest_hash", maintain_order=True)
        .head(top_n)
    )

In [ ]:
top_combos = _resolve_pre_risk_runs(
    CASE_STUDY_ID,
    LABEL,
    split="validation",
    top_n=TOP_N_COMBOS,
)

if top_combos.is_empty():
    msg = "No baseline or allocation results found. Run the upstream notebooks first."
    raise RuntimeError(msg)

for row in top_combos.iter_rows(named=True):
    spec = json.loads(row["spec_json"])
    alloc = strategy_view(spec).get("allocation", {}).get("method", "equal_weight")
    print(f"  Sharpe={row['sharpe']:.3f}  alloc={alloc}  bt_hash={row['backtest_hash'][:8]}")

In [ ]:
prices = load_backtest_prices_for(CASE_STUDY_ID, LABEL, split="validation", max_symbols=MAX_SYMBOLS)

### MAE/MFE-Calibrated Trailing Stops

MAE/MFE calibration requires an engine path with intra-period prices. The
vectorized monthly-outcome path skips this calibration and leaves the
configured position-control catalog unexecuted.

In [ ]:
_position_grid = get_position_risk_controls(CASE_STUDY_ID)
if not IS_VECTORIZED and "close" in prices.columns:
    calibrated = calibrate_trailing_stops(prices)
    if calibrated:
        existing_thresholds = {rc.get("threshold", 0) for rc in _position_grid}
        new_calibrated = [c for c in calibrated if c["threshold"] not in existing_thresholds]
        position_controls = _position_grid + new_calibrated
        print(f"MAE/MFE calibration added {len(new_calibrated)} thresholds")
    else:
        position_controls = _position_grid
        print("MAE/MFE calibration returned no results; using standard grid")
else:
    position_controls = _position_grid
    print("Skipping MAE/MFE calibration (vectorized or no close column)")

portfolio_controls = get_portfolio_risk_controls(CASE_STUDY_ID)
# Portfolio-limit overlays were purged 2026-05-17; this CS sweeps position-level
# overlays only. Fail loudly if a portfolio overlay is ever re-introduced into
# setup.yaml so it cannot silently re-file overlay backtests against the spine.
assert not portfolio_controls, (
    f"Unexpected portfolio risk controls for {CASE_STUDY_ID}: {portfolio_controls}. "
    "Portfolio-limit overlays were removed; only position-level overlays are swept."
)
if MAX_RISK_VARIANTS > 0:
    position_controls = position_controls[:MAX_RISK_VARIANTS]
    portfolio_controls = portfolio_controls[:MAX_RISK_VARIANTS]
    print(f"Risk variants limited to {MAX_RISK_VARIANTS} each")

## 2. Risk Overlay Sweep

On an engine-path case study this loop registers one backtest per position-level
control. Here the position loop is skipped because the path cannot represent the
rules, and the portfolio-control list is empty by configuration, so the loop body
has nothing to register and the count below is zero by construction rather than by
failure. The two are different outcomes and the counters separate them.

In [ ]:
n_done = 0
n_failed = 0

# Every run inside this loop is fed `combo_weights`, and computing them means running
# the parent's allocator again - MVO and HRP take minutes. If neither control list can
# produce a run, that work has no consumer, so it is not started. Without this the
# notebook pays the full allocator cost to register nothing.
will_register = bool(portfolio_controls) or (not IS_VECTORIZED and bool(position_controls))
if not will_register:
    print(
        "No control can run on this backtest path, so no allocation weights are "
        "computed and no backtest is registered."
    )

for combo_idx, combo_row in enumerate(top_combos.iter_rows(named=True) if will_register else []):
    pred_hash = combo_row["prediction_hash"]
    base_spec = ensure_backtest_spec(
        CASE_STUDY_ID,
        bt_config,
        json.loads(combo_row["spec_json"]),
        prices=prices,
        prediction_hash=pred_hash,
        initial_cash=bt_config.initial_cash,
    )
    alloc_method = strategy_view(base_spec).get("allocation", {}).get("method", "equal_weight")

    predictions = read_predictions(CASE_STUDY_ID, pred_hash)

    t0 = time.time()
    combo_weights = precompute_weights(
        predictions, base_spec, prices, label=LABEL, case_study=CASE_STUDY_ID
    )
    print(
        f"  Combo {combo_idx + 1}/{len(top_combos)}: {alloc_method} - "
        f"weights precomputed in {time.time() - t0:.0f}s"
    )

    # Position-level risk rules (engine only)
    if not IS_VECTORIZED:
        for rc in position_controls:
            spec_risk = clone_backtest_spec(base_spec)
            spec_risk["chapter"] = "ch19"
            if rc["type"] == "time_exit":
                spec_risk["strategy"]["risk"] = {
                    "name": rc["name"],
                    "position_rules": [{"type": rc["type"], "bars": rc["bars"]}],
                }
            else:
                spec_risk["strategy"]["risk"] = {
                    "name": rc["name"],
                    "position_rules": [{"type": rc["type"], "threshold": rc["threshold"]}],
                }

            try:
                result = run_backtest(
                    CASE_STUDY_ID,
                    pred_hash,
                    spec_risk,
                    prices=prices,
                    predictions=predictions,
                    label=LABEL,
                    register=True,
                    initial_cash=bt_config.initial_cash,
                    calendar=bt_config.calendar,
                    precomputed_weights=combo_weights,
                )
                n_done += 1
                print(
                    f"    {rc['name']}: Sharpe={result.metrics.get('sharpe', 0):.3f}, "
                    f"MaxDD={result.metrics.get('max_drawdown', 0):.2%}"
                )
            except Exception as e:
                n_failed += 1
                print(f"    {rc['name']}: FAILED - {e}")

    # Portfolio-level risk limits
    for rc in portfolio_controls:
        spec_risk = clone_backtest_spec(base_spec)
        spec_risk["chapter"] = "ch19"
        spec_risk["strategy"]["risk"] = {
            "name": rc["name"],
            "portfolio_limits": [{"type": rc["type"], "threshold": rc["threshold"]}],
        }

        try:
            result = run_backtest(
                CASE_STUDY_ID,
                pred_hash,
                spec_risk,
                prices=prices,
                predictions=predictions,
                label=LABEL,
                register=True,
                initial_cash=bt_config.initial_cash,
                calendar=bt_config.calendar,
                precomputed_weights=combo_weights,
            )
            n_done += 1
            print(
                f"    {rc['name']}: Sharpe={result.metrics.get('sharpe', 0):.3f}, "
                f"MaxDD={result.metrics.get('max_drawdown', 0):.2%}"
            )
        except Exception as e:
            n_failed += 1
            print(f"    {rc['name']}: FAILED - {e}")

print(f"\nRisk sweep complete: {n_done} registered, {n_failed} failed")

## 3. What The Registry Holds

This section only reads. It asks the registry for every overlay run filed against
this case study and, for each, the change in Sharpe against the parent it was
applied to.

An empty answer here is the point of the notebook rather than a gap in it. The
alternative - running the controls anyway on invented intra-month prices - would
put a Sharpe delta next to each rule, and a reader would have no way to tell that
number from one a stop had actually earned.

In [ ]:
explorer = BacktestExplorer(CASE_STUDY_ID)

In [ ]:
risk_df = explorer.risk_impact()

if not risk_df.is_empty():
    # Best by risk type
    for risk_type in risk_df["risk_type"].unique().sort().to_list():
        subset = risk_df.filter(pl.col("risk_type") == risk_type).sort("sharpe", descending=True)
        best = subset.head(1)
        print(f"  Best {risk_type}: {best['risk_name'][0]} → Sharpe={best['sharpe'][0]:.3f}")

    print(f"\nAll risk overlays ({len(risk_df)}):")
    print(
        risk_df.select("risk_name", "risk_type", "sharpe", "max_drawdown", "sharpe_delta")
        .sort("sharpe", descending=True)
        .head(15)
    )
else:
    print("No risk overlay data in registry")

## Key Takeaways

1. Whether a rule can be represented is a property of the backtest path, not a
   setting. A stop needs a price between rebalances; the vectorized forward-return
   path holds one return per rebalance and has none, so a stop cannot be evaluated
   on it at any parameter value.
2. The configuration still declares the position-level controls, because the same
   file drives the engine-path case studies where they do run. Declared and
   applicable are separate questions, and this notebook answers the second.
3. Portfolio-level limits are absent on purpose. A gross-exposure or per-name cap
   is a constraint the desk operates under, not a variant that competes for the
   highest validation Sharpe, and sweeping it as one invites keeping whichever cap
   was loosest on the grounds that it scored highest.
4. Nothing was registered and the sealed holdout was not read, so the funnel enters
   the strategy analysis with the parent from section 1 unchanged.

**Next:** `15_strategy_analysis` confronts the selection this funnel performed and
is where the results are interpreted.